
# Exp5 — Third-Architecture Functional Reliance
## Cross-channel TimeMixer confirmation for ICLR 2027

### Purpose

> Is the forecaster-dependence of functional source reliance specific to only iTransformer and TimesNet?

This experiment adds a **third neural forecasting architecture** while keeping the intervention protocol fixed.

The existing Rethinking study already evaluates functional reliance for:

- iTransformer
- TimesNet

using the same three preselected conditions:

- Electricity, \(H=192\)
- Solar, \(H=720\)
- ETTh1, \(H=720\)

with the same target–source pairs, 512 test windows, and 16 donor mappings.

This notebook adds **TimeMixer with explicit cross-channel mixing**
(`channel_independence = 0`) and reuses **exactly the same** EPI protocol.

### Why this variant?

The experiment is intended to test a scientific question, not to search architectures after observing outcomes.

A channel-independent forecaster would make source-channel EPI trivially near zero, so the predeclared third model is the standard TimeMixer backbone configured with **cross-channel mixing enabled**.

No architecture, candidate pool, donor mapping, target set, threshold, or metric is tuned after observing the result.

### Primary metric

Full-horizon MSE EPI:

\[
I^{\mathcal F}_{ij}
=
100\,
\frac{
\mathbb E[\ell(\mathcal F(X_{j\leftarrow \pi(j)}),y)]
-
\mathbb E[\ell(\mathcal F(X),y)]
}{
\mathbb E[\ell(\mathcal F(X),y)]
}.
\]

### Primary questions

1. Does TimeMixer functionally use cross-channel information?
2. Is TimeMixer EPI reproducible under donor and random-window split halves?
3. How strongly does TimeMixer source reliance align with:
   - controlled predictive utility \(P\),
   - iTransformer,
   - TimesNet?
4. After adding TimeMixer, is there evidence for a single architecture-independent functional source graph?

### Run protocol

- `RUN_MODE="screen"`: ETTh1 H=720 only. Validates import, training, checkpoint, cross-channel effect, and EPI cache.
- `RUN_MODE="full"`: Electricity H=192, Solar H=720, ETTh1 H=720.

Run the screen first. If it executes correctly, run the full suite **regardless of the sign of the screen result**.


In [1]:

from pathlib import Path
from types import SimpleNamespace
import copy
import gc
import importlib
import itertools
import math
import os
import random
import sys
import time
import traceback
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Python: 3.11.16 (main, Aug 25 2026, 14:00:53) [Clang 22.1.3 ]
PyTorch: 2.5.1+cu121
Device: cuda
GPU: NVIDIA A100-SXM4-80GB


### v2 — independent-server fix

The Rethinking experiments run on a different GPU server, so this version no longer assumes that `Time-Series-Library` is already installed.

The notebook now:

1. checks several known locations;
2. performs a shallow `/data` search for `models/TimeMixer.py`;
3. if absent, automatically clones `thuml/Time-Series-Library` under  
   `/data/code/2026_08/external/Time-Series-Library`;
4. records the exact Git commit hash for reproducibility;
5. reports a clear requirements-install command if a Python dependency is missing.

No scientific protocol or EPI metric was changed.


## 1. Fixed configuration

In [2]:

SEED = 2026

SEQ_LEN = 96
LABEL_LEN = 0

ALL_CONDITIONS = [
    ("Electricity", 192),
    ("Solar", 720),
    ("ETTh1", 720),
]

RUN_MODE = "full"  # "screen" or "full"

CONDITIONS = (
    [("ETTh1", 720)]
    if RUN_MODE == "screen"
    else ALL_CONDITIONS
)

PROJECT_ROOT = Path("/data/code/2026_08")

PHASE1_ROOT = PROJECT_ROOT / "results_external_baselines_phase1_direct"

ITR_EPI_ROOT = (
    PROJECT_ROOT
    / "results_expected_permutation_importance_reliability"
)
ITR_EPI_PAIR_FILE = (
    ITR_EPI_ROOT
    / "expected_permutation_pair_scores.csv"
)

TIMESNET_EPI_ROOT = (
    PROJECT_ROOT
    / "results_timesnet_strong_backbone_epi"
)
TIMESNET_PAIR_FILE = (
    TIMESNET_EPI_ROOT
    / "timesnet_epi_pair_scores.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results_timemixer_crosschannel_epi"
)
CKPT_ROOT = OUTPUT_DIR / "checkpoints"
CACHE_DIR = OUTPUT_DIR / "source_delta_cache"

for p in [OUTPUT_DIR, CKPT_ROOT, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Fixed before seeing results.
TRAIN_EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
NUM_WORKERS = 2

# Cross-channel TimeMixer config.
# channel_independence=0 is essential for the scientific question.
TIMEMIXER_CFG = {
    "e_layers": 2,
    "d_layers": 1,
    "d_model": 32,
    "d_ff": 32,
    "n_heads": 8,
    "factor": 1,
    "dropout": 0.1,
    "moving_avg": 25,
    "down_sampling_layers": 3,
    "down_sampling_window": 2,
    "down_sampling_method": "avg",
    "decomp_method": "moving_avg",
    "channel_independence": 0,
    "use_norm": 1,
}

TRAIN_BATCH = {
    "Electricity": 16,
    "Solar": 16,
    "ETTh1": 32,
}

EPI_BATCH = {
    "Electricity": 4,
    "Solar": 4,
    "ETTh1": 16,
}

DONOR_CHUNK = {
    "Electricity": 2,
    "Solar": 2,
    "ETTh1": 4,
}

N_SPLITS = 50
N_TEMPORAL_BLOCKS = 4
RANK_K = 5

print("RUN_MODE:", RUN_MODE)
print("Conditions:", CONDITIONS)
print("Output:", OUTPUT_DIR)


RUN_MODE: full
Conditions: [('Electricity', 192), ('Solar', 720), ('ETTh1', 720)]
Output: /data/code/2026_08/results_timemixer_crosschannel_epi


## 2. Determinism and ranking helpers

In [3]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def safe_spearman(a, b):
    d = (
        pd.DataFrame({"a": a, "b": b})
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    if len(d) < 3:
        return np.nan
    if d["a"].nunique() < 2 or d["b"].nunique() < 2:
        return np.nan
    return float(d.corr(method="spearman").iloc[0, 1])

def topk_set(scores, ids, k):
    scores = np.asarray(scores, dtype=np.float64)
    ids = np.asarray(ids, dtype=np.int64)
    order = np.argsort(scores)[::-1][:k]
    return set(ids[order].tolist())

def topk_jaccard(a, b, ids, k):
    A = topk_set(a, ids, k)
    B = topk_set(b, ids, k)
    return len(A & B) / max(1, len(A | B))


## 3. Reuse the exact existing iTransformer / TimesNet EPI protocol

In [4]:

if not ITR_EPI_PAIR_FILE.exists():
    raise FileNotFoundError(
        f"Missing iTransformer EPI pair file:\n{ITR_EPI_PAIR_FILE}\n"
        "Run Expected_Permutation_Importance_Reliability_ICLR27.ipynb first."
    )

previous = pd.read_csv(ITR_EPI_PAIR_FILE)

required = [
    "dataset", "pred_len", "target_channel", "source_channel", "pool_tag",
    "P_endpoint_mse_%", "P_full_mse_%", "P_full_mae_%",
    "I_Ridge_endpoint_mse_%", "I_Ridge_full_mse_%", "I_Ridge_full_mae_%",
    "I_MLP_endpoint_mse_%", "I_MLP_full_mse_%", "I_MLP_full_mae_%",
    "EPI_endpoint_mse_%", "EPI_full_mse_%", "EPI_full_mae_%",
]

missing = [c for c in required if c not in previous.columns]
if missing:
    raise RuntimeError(f"Prior EPI CSV missing columns: {missing}")

wanted = set(CONDITIONS)
previous = previous[
    previous.apply(
        lambda r: (r["dataset"], int(r["pred_len"])) in wanted,
        axis=1,
    )
].copy()

print("Matched iTransformer pair rows:", len(previous))

if TIMESNET_PAIR_FILE.exists():
    timesnet_previous = pd.read_csv(TIMESNET_PAIR_FILE)
    timesnet_previous = timesnet_previous[
        timesnet_previous.apply(
            lambda r: (r["dataset"], int(r["pred_len"])) in wanted,
            axis=1,
        )
    ].copy()
    print("Matched TimesNet pair rows:", len(timesnet_previous))
else:
    timesnet_previous = None
    print("TimesNet pair file not found; TimeMixer vs TimesNet comparison will be skipped.")


Matched iTransformer pair rows: 393
Matched TimesNet pair rows: 393


## 4. Locate Time-Series-Library and import TimeMixer

In [5]:

import subprocess
import importlib
import importlib.util

# ---------------------------------------------------------------------
# Locate or fetch Time-Series-Library.
# This Rethinking server is independent from the independent execution server,
# so Time-Series-Library may not already be installed.
# ---------------------------------------------------------------------

TSLIB_CANDIDATES = [
    Path("/data/Time-Series-Library_v2"),
    Path("/data/Time-Series-Library"),
    Path("/data/time-series-foundation-model/Time-Series-Library"),
    Path("/data/time-series_foundation_model/Time-Series-Library"),
    PROJECT_ROOT / "Time-Series-Library",
    PROJECT_ROOT / "external" / "Time-Series-Library",
]

AUTO_CLONE_TSLIB = True
TSLIB_INSTALL_DIR = PROJECT_ROOT / "external" / "Time-Series-Library"

def has_timemixer(root):
    root = Path(root)
    return (root / "models" / "TimeMixer.py").exists()

def locate_timemixer_repo():
    # 1) Fast known-path check.
    for p in TSLIB_CANDIDATES:
        if has_timemixer(p):
            return p.resolve()

    # 2) Shallow system search in /data (avoids a huge recursive Python walk).
    try:
        result = subprocess.run(
            [
                "find", "/data",
                "-maxdepth", "6",
                "-type", "f",
                "-path", "*/models/TimeMixer.py",
                "-print", "-quit",
            ],
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )
        found = result.stdout.strip()
        if found:
            return Path(found).resolve().parents[1]
    except Exception as e:
        print("Shallow /data search skipped:", repr(e))

    return None

TSL_ROOT = locate_timemixer_repo()

if TSL_ROOT is None and AUTO_CLONE_TSLIB:
    TSLIB_INSTALL_DIR.parent.mkdir(parents=True, exist_ok=True)

    if TSLIB_INSTALL_DIR.exists() and not has_timemixer(TSLIB_INSTALL_DIR):
        raise RuntimeError(
            f"{TSLIB_INSTALL_DIR} exists but does not contain models/TimeMixer.py. "
            "Rename/remove that directory or set TSLIB_INSTALL_DIR to another path."
        )

    if not TSLIB_INSTALL_DIR.exists():
        print("Time-Series-Library not found on this server.")
        print("Cloning THUML/Time-Series-Library ...")
        cmd = [
            "git", "clone", "--depth", "1",
            "https://github.com/thuml/Time-Series-Library.git",
            str(TSLIB_INSTALL_DIR),
        ]
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=False,
        )
        print(proc.stdout)
        if proc.returncode != 0:
            print(proc.stderr)
            raise RuntimeError(
                "Automatic clone failed. "
                "Please clone https://github.com/thuml/Time-Series-Library.git "
                f"manually into {TSLIB_INSTALL_DIR}, then rerun this cell."
            )

    TSL_ROOT = locate_timemixer_repo()

if TSL_ROOT is None:
    raise FileNotFoundError(
        "Could not locate Time-Series-Library with models/TimeMixer.py. "
        "Set AUTO_CLONE_TSLIB=True or edit TSLIB_CANDIDATES."
    )

# Put the chosen repository first so `models.*` resolves to this checkout.
tsl_str = str(TSL_ROOT)
if tsl_str in sys.path:
    sys.path.remove(tsl_str)
sys.path.insert(0, tsl_str)

# Avoid accidentally reusing a `models` package imported from another repo.
for modname in list(sys.modules):
    if modname == "models" or modname.startswith("models."):
        del sys.modules[modname]
importlib.invalidate_caches()

try:
    TimeMixer_module = importlib.import_module("models.TimeMixer")
except ModuleNotFoundError as e:
    print("TimeMixer import failed because a Python dependency is missing:")
    print(" ", repr(e))
    print()
    print("Repository:", TSL_ROOT)
    print("Install the repository requirements, e.g.:")
    print(f"  pip install -r {TSL_ROOT / 'requirements.txt'}")
    print("Then restart the kernel and rerun from the top.")
    raise

# Record the exact checkout for reproducibility.
try:
    TSLIB_COMMIT = subprocess.check_output(
        ["git", "-C", str(TSL_ROOT), "rev-parse", "HEAD"],
        text=True,
    ).strip()
except Exception:
    TSLIB_COMMIT = "unknown"

print("TSLib:", TSL_ROOT)
print("TSLib commit:", TSLIB_COMMIT)
print("TimeMixer module:", TimeMixer_module)


TSLib: /data/code/forecast_jepa/Time-Series-Library
TSLib commit: 4e938a1767106324dd753b2a44832bf870a0252e
TimeMixer module: <module 'models.TimeMixer' from '/data/code/forecast_jepa/Time-Series-Library/models/TimeMixer.py'>


## 5. Dataset paths and identical train-only normalization

In [6]:

DATASET_CANDIDATES = {
    "Electricity": [
        "/data/dataset/electricity/electricity.csv",
        "/data/dataset/ECL/electricity.csv",
        "/data/dataset/electricity.csv",
    ],
    "Solar": [
        "/data/dataset/solar/solar_AL.txt",
        "/data/dataset/Solar/solar_AL.txt",
        "/data/dataset/solar_AL.txt",
    ],
    "ETTh1": [
        "/data/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETTh1.csv",
    ],
}

DATASET_SPECS = {
    "Electricity": {"channels": 321, "freq": "h", "data": "custom", "target": "OT"},
    "Solar": {"channels": 137, "freq": "h", "data": "Solar", "target": "OT"},
    "ETTh1": {"channels": 7, "freq": "h", "data": "ETTh1", "target": "OT"},
}

def resolve_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

DATASET_FILES = {}
for d, candidates in DATASET_CANDIDATES.items():
    p = resolve_path(candidates)
    if p is None:
        raise FileNotFoundError(f"{d}: dataset file not found.")
    DATASET_FILES[d] = p
    print(f"{d:12s} -> {p}")

def load_numeric_series(dataset_name, path):
    path = Path(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

        date_col = None
        for candidate in ["date", "datetime", "timestamp", "time"]:
            if candidate in df.columns:
                date_col = candidate
                break

        if date_col is None:
            first = df.columns[0]
            if not pd.api.types.is_numeric_dtype(df[first]):
                date_col = first

        if date_col is not None:
            df = df.drop(columns=[date_col])

        df = df.select_dtypes(include=[np.number])
        x = df.to_numpy(dtype=np.float32)

    else:
        try:
            x = np.loadtxt(path, delimiter=",", dtype=np.float32)
        except Exception:
            x = np.loadtxt(path, dtype=np.float32)

        if x.ndim == 1:
            x = x[:, None]

    if x.ndim != 2:
        raise ValueError(f"{dataset_name}: unexpected shape {x.shape}")
    if not np.isfinite(x).all():
        raise ValueError(f"{dataset_name}: non-finite values")

    return x

def split_boundaries(dataset_name, T):
    if dataset_name == "ETTh1":
        train_end = 12 * 30 * 24
        val_end = train_end + 4 * 30 * 24
        test_end = val_end + 4 * 30 * 24
        return min(train_end, T), min(val_end, T), min(test_end, T)

    return int(0.70 * T), int(0.80 * T), T

def load_and_normalize(dataset_name):
    raw = load_numeric_series(dataset_name, DATASET_FILES[dataset_name])
    T, C = raw.shape

    train_end, val_end, test_end = split_boundaries(dataset_name, T)

    train = raw[:train_end]
    mean = train.mean(axis=0, keepdims=True)
    std = np.maximum(train.std(axis=0, keepdims=True), 1e-6)

    x = (raw - mean) / std

    return x.astype(np.float32), {
        "T": T,
        "C": C,
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }


Electricity  -> /data/dataset/electricity/electricity.csv
Solar        -> /data/dataset/solar/solar_AL.txt
ETTh1        -> /data/dataset/ETT-small/ETTh1.csv


## 6. Training dataset and TimeMixer arguments

In [7]:

class ForecastDataset(Dataset):
    def __init__(
        self,
        data,
        seq_len,
        pred_len,
        target_start_min,
        target_end_exclusive,
    ):
        self.data = torch.from_numpy(data).float()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)

        first_t = max(self.seq_len, int(target_start_min))
        last_t = int(target_end_exclusive) - self.pred_len

        self.origins = np.arange(
            first_t,
            last_t + 1,
            dtype=np.int64,
        )

        if len(self.origins) == 0:
            raise ValueError("No valid forecasting windows.")

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        t = int(self.origins[idx])
        x = self.data[t-self.seq_len:t]
        y = self.data[t:t+self.pred_len]
        return x, y

def build_timemixer_args(dataset_name, H):
    spec = DATASET_SPECS[dataset_name]
    cfg = TIMEMIXER_CFG
    data_file = DATASET_FILES[dataset_name]

    return SimpleNamespace(
        task_name="long_term_forecast",
        model="TimeMixer",

        data=spec["data"],
        root_path=str(data_file.parent) + "/",
        data_path=data_file.name,
        features="M",
        target=spec["target"],
        freq=spec["freq"],

        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=int(H),
        seasonal_patterns="Monthly",
        inverse=False,

        enc_in=spec["channels"],
        dec_in=spec["channels"],
        c_out=spec["channels"],

        e_layers=cfg["e_layers"],
        d_layers=cfg["d_layers"],
        factor=cfg["factor"],
        d_model=cfg["d_model"],
        n_heads=cfg["n_heads"],
        d_ff=cfg["d_ff"],
        moving_avg=cfg["moving_avg"],
        distil=True,
        dropout=cfg["dropout"],
        embed="timeF",
        activation="gelu",
        output_attention=False,

        channel_independence=cfg["channel_independence"],
        decomp_method=cfg["decomp_method"],
        use_norm=cfg["use_norm"],
        down_sampling_layers=cfg["down_sampling_layers"],
        down_sampling_window=cfg["down_sampling_window"],
        down_sampling_method=cfg["down_sampling_method"],
        use_future_temporal_feature=0,

        top_k=5,
        num_kernels=6,

        class_strategy="projection",
        augmentation_ratio=0,
    )

def build_timemixer(dataset_name, H):
    args = build_timemixer_args(dataset_name, H)
    model = TimeMixer_module.Model(args).float().to(DEVICE)
    return model, args

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Construction pre-flight.
for d, H in CONDITIONS:
    model, args = build_timemixer(d, H)
    print(
        f"{d} H={H} | params={count_parameters(model):,} | "
        f"channel_independence={args.channel_independence}"
    )
    if int(args.channel_independence) != 0:
        raise RuntimeError("This experiment requires channel_independence=0.")
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


Electricity H=192 | params=191,197 | channel_independence=0
Solar H=720 | params=360,293 | channel_independence=0
ETTh1 H=720 | params=342,483 | channel_independence=0


## 7. Forward pass, training, checkpoints

In [8]:

def timemixer_forward(model, x, H):
    B, _, C = x.shape

    dec_inp = torch.zeros(
        B, H, C,
        dtype=x.dtype,
        device=x.device,
    )

    out = model(x, None, dec_inp, None)

    if isinstance(out, tuple):
        out = out[0]

    return out[:, -H:, :]

def condition_dir(dataset_name, H):
    p = CKPT_ROOT / dataset_name / f"H{H}"
    p.mkdir(parents=True, exist_ok=True)
    return p

def ckpt_path(dataset_name, H):
    return condition_dir(dataset_name, H) / "best_model.pt"

def result_path(dataset_name, H):
    return condition_dir(dataset_name, H) / "result.csv"

def history_path(dataset_name, H):
    return condition_dir(dataset_name, H) / "history.csv"

def make_loaders(dataset_name, H):
    x, meta = load_and_normalize(dataset_name)

    if meta["C"] != DATASET_SPECS[dataset_name]["channels"]:
        raise RuntimeError(
            f"{dataset_name}: channel mismatch {meta['C']} "
            f"vs {DATASET_SPECS[dataset_name]['channels']}"
        )

    train_ds = ForecastDataset(
        x, SEQ_LEN, H,
        target_start_min=SEQ_LEN,
        target_end_exclusive=meta["train_end"],
    )
    val_ds = ForecastDataset(
        x, SEQ_LEN, H,
        target_start_min=meta["train_end"],
        target_end_exclusive=meta["val_end"],
    )
    test_ds = ForecastDataset(
        x, SEQ_LEN, H,
        target_start_min=meta["val_end"],
        target_end_exclusive=meta["test_end"],
    )

    g = torch.Generator()
    g.manual_seed(SEED + 100000)

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

    batch = TRAIN_BATCH[dataset_name]

    train_loader = DataLoader(
        train_ds,
        batch_size=batch,
        shuffle=True,
        generator=g,
        **common,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch,
        shuffle=False,
        **common,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch,
        shuffle=False,
        **common,
    )

    return x, meta, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

@torch.no_grad()
def evaluate_model(model, loader, H):
    model.eval()

    total_sse = 0.0
    total_sae = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.float().to(DEVICE, non_blocking=True)
        yb = yb.float().to(DEVICE, non_blocking=True)

        pred = timemixer_forward(model, xb, H)
        err = pred - yb

        total_sse += err.square().sum().item()
        total_sae += err.abs().sum().item()
        total_n += err.numel()

    return total_sse / total_n, total_sae / total_n

def lr_for_epoch(epoch):
    return LEARNING_RATE * (0.5 ** ((epoch - 1) // 5))

def train_or_load_timemixer(dataset_name, H):
    cpath = ckpt_path(dataset_name, H)
    rpath = result_path(dataset_name, H)

    if cpath.exists() and rpath.exists():
        model, args = build_timemixer(dataset_name, H)
        state = torch.load(cpath, map_location="cpu")
        model.load_state_dict(state, strict=True)
        model.eval()
        result = pd.read_csv(rpath).iloc[0].to_dict()
        print(
            f"Reusing checkpoint: {dataset_name} H={H} | "
            f"test MSE={float(result['test_mse']):.6f}"
        )
        return model, args, result

    set_seed(SEED)

    (
        x, meta,
        train_ds, val_ds, test_ds,
        train_loader, val_loader, test_loader,
    ) = make_loaders(dataset_name, H)

    model, args = build_timemixer(dataset_name, H)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    bad_epochs = 0
    history = []
    start_time = time.time()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        lr = lr_for_epoch(epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr

        model.train()

        train_sse = 0.0
        train_n = 0

        for xb, yb in train_loader:
            xb = xb.float().to(DEVICE, non_blocking=True)
            yb = yb.float().to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred = timemixer_forward(model, xb, H)
            loss = F.mse_loss(pred, yb)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss: {dataset_name} H={H}"
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_sse += (
                (pred.detach() - yb).square().sum().item()
            )
            train_n += yb.numel()

        train_mse = train_sse / train_n
        val_mse, val_mae = evaluate_model(model, val_loader, H)

        history.append({
            "epoch": epoch,
            "learning_rate": lr,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "val_mae": val_mae,
        })

        print(
            f"TimeMixer-XC | {dataset_name:11s} | H={H:3d} | "
            f"epoch={epoch:02d} | train={train_mse:.6f} | "
            f"val={val_mse:.6f} | lr={lr:.2e}"
        )

        if val_mse < best_val:
            best_val = val_mse
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    if best_state is None:
        raise RuntimeError("No best checkpoint.")

    model.load_state_dict(best_state)
    torch.save(best_state, cpath)

    test_mse, test_mae = evaluate_model(model, test_loader, H)

    result = {
        "model": "TimeMixer-XC",
        "dataset": dataset_name,
        "pred_len": int(H),
        "seed": SEED,
        "seq_len": SEQ_LEN,
        "label_len": LABEL_LEN,
        "channel_independence": 0,
        "best_epoch": best_epoch,
        "best_val_mse": best_val,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "parameters": count_parameters(model),
        "runtime_sec": time.time() - start_time,
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
        "test_samples": len(test_ds),
    }

    pd.DataFrame([result]).to_csv(rpath, index=False)
    pd.DataFrame(history).to_csv(history_path(dataset_name, H), index=False)

    print(
        f"TEST {dataset_name} H={H}: "
        f"MSE={test_mse:.6f} MAE={test_mae:.6f} "
        f"best_epoch={best_epoch}"
    )

    del (
        train_loader, val_loader, test_loader,
        train_ds, val_ds, test_ds, x
    )
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    model.eval()
    return model, args, result


## 8. Train / load the third backbone

In [9]:

training_rows = []

for d, H in CONDITIONS:
    print("\n" + "=" * 100)
    print(f"TRAIN / LOAD TimeMixer-XC | {d} H={H}")
    print("=" * 100)

    model, args, result = train_or_load_timemixer(d, H)
    training_rows.append(result)

    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

training_summary = pd.DataFrame(training_rows)
training_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_timemixer_xc_training_summary.csv",
    index=False,
)
display(training_summary)



TRAIN / LOAD TimeMixer-XC | Electricity H=192
TimeMixer-XC | Electricity | H=192 | epoch=01 | train=0.232524 | val=0.171751 | lr=1.00e-03
TimeMixer-XC | Electricity | H=192 | epoch=02 | train=0.170162 | val=0.159981 | lr=1.00e-03
TimeMixer-XC | Electricity | H=192 | epoch=03 | train=0.155649 | val=0.162786 | lr=1.00e-03
TimeMixer-XC | Electricity | H=192 | epoch=04 | train=0.147201 | val=0.148284 | lr=1.00e-03
TimeMixer-XC | Electricity | H=192 | epoch=05 | train=0.141620 | val=0.152388 | lr=1.00e-03
TimeMixer-XC | Electricity | H=192 | epoch=06 | train=0.136072 | val=0.150477 | lr=5.00e-04
TimeMixer-XC | Electricity | H=192 | epoch=07 | train=0.133877 | val=0.148328 | lr=5.00e-04
TimeMixer-XC | Electricity | H=192 | epoch=08 | train=0.131859 | val=0.145760 | lr=5.00e-04
TimeMixer-XC | Electricity | H=192 | epoch=09 | train=0.130147 | val=0.149250 | lr=5.00e-04
TimeMixer-XC | Electricity | H=192 | epoch=10 | train=0.128683 | val=0.146614 | lr=5.00e-04
TimeMixer-XC | Electricity | H=19

,model,dataset,pred_len,seed,seq_len,label_len,channel_independence,best_epoch,best_val_mse,test_mse,test_mae,parameters,runtime_sec,train_samples,val_samples,test_samples
0,TimeMixer-XC,Electricity,192,2026,96,0,0,14,0.143080,0.165758,0.266455,191197,657.325047,18125,2440,5070
1,TimeMixer-XC,Solar,720,2026,96,0,0,2,0.207883,0.279617,0.306191,360293,457.011552,35977,4537,9793
2,TimeMixer-XC,ETTh1,720,2026,96,0,0,2,1.655084,0.565757,0.535723,342483,62.960510,7825,2161,2161


## 9. Load the exact existing EPI origins, targets, sources, and donor maps

In [10]:

def load_condition_protocol(dataset_name, H):
    meta_path = ITR_EPI_ROOT / f"{dataset_name}_H{H}_meta.npz"

    if not meta_path.exists():
        raise FileNotFoundError(
            f"Missing existing EPI protocol: {meta_path}"
        )

    z = np.load(meta_path)

    protocol = {
        "origins": z["origins"].astype(np.int64),
        "targets": z["targets"].astype(np.int64),
        "sources": z["sources"].astype(np.int64),
        "shifts": z["shifts"].astype(np.int64),
        "maps": z["maps"].astype(np.int64),
    }

    if protocol["maps"].shape[1] != len(protocol["origins"]):
        raise RuntimeError("Donor map / origin mismatch.")

    return protocol

for d, H in CONDITIONS:
    p = load_condition_protocol(d, H)
    pair_df = previous[
        (previous["dataset"] == d)
        & (previous["pred_len"].astype(int) == int(H))
    ]

    print(
        f"{d} H={H}: windows={len(p['origins'])}, "
        f"targets={len(p['targets'])}, sources={len(p['sources'])}, "
        f"donors={len(p['shifts'])}, pairs={len(pair_df)}"
    )


Electricity H=192: windows=512, targets=6, sources=132, donors=16, pairs=173
Solar H=720: windows=512, targets=6, sources=94, donors=16, pairs=178
ETTh1 H=720: windows=512, targets=7, sources=7, donors=16, pairs=42


## 10. Clean loss and grouped all-other cross-channel usage check

In [11]:

def make_XY_from_origins(x, origins, H):
    X = np.stack(
        [x[t-SEQ_LEN:t] for t in origins],
        axis=0,
    ).astype(np.float32)

    Y = np.stack(
        [x[t:t+H] for t in origins],
        axis=0,
    ).astype(np.float32)

    return X, Y

@torch.inference_mode()
def clean_losses(model, X, Y, targets, H, batch_size):
    N = len(X)
    Tn = len(targets)

    output = {
        "endpoint_mse": np.zeros((N, Tn), dtype=np.float32),
        "full_mse": np.zeros((N, Tn), dtype=np.float32),
        "full_mae": np.zeros((N, Tn), dtype=np.float32),
    }

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        xb = torch.from_numpy(X[start:end]).float().to(DEVICE)
        yb = torch.from_numpy(Y[start:end]).float().to(DEVICE)

        pred = timemixer_forward(model, xb, H)

        err = pred[:, :, targets] - yb[:, :, targets]

        output["endpoint_mse"][start:end] = (
            err[:, -1, :].square().cpu().numpy()
        )
        output["full_mse"][start:end] = (
            err.square().mean(dim=1).cpu().numpy()
        )
        output["full_mae"][start:end] = (
            err.abs().mean(dim=1).cpu().numpy()
        )

    return output

@torch.inference_mode()
def grouped_all_other_gain(
    model,
    X,
    Y,
    targets,
    donor_maps,
    H,
    batch_size,
    max_donors=4,
):
    donors = np.arange(min(max_donors, donor_maps.shape[0]))
    rows = []

    clean = clean_losses(
        model, X, Y, targets, H, batch_size
    )["full_mse"]

    C = X.shape[2]

    for tq, target in enumerate(targets):
        target = int(target)

        deltas = []

        for r in donors:
            pert_losses = []

            for start in range(0, len(X), batch_size):
                end = min(start + batch_size, len(X))

                base = X[start:end].copy()
                global_idx = np.arange(start, end, dtype=np.int64)
                donor_idx = donor_maps[r, global_idx]

                # Replace every non-target source history simultaneously.
                all_ch = np.arange(C)
                src_ch = all_ch[all_ch != target]

                donor_block = X[donor_idx][:, :, src_ch]
                base[:, :, src_ch] = donor_block

                xb = torch.from_numpy(base).float().to(DEVICE)
                yb = torch.from_numpy(Y[start:end]).float().to(DEVICE)

                pred = timemixer_forward(model, xb, H)
                err = pred[:, :, target] - yb[:, :, target]
                loss = err.square().mean(dim=1).cpu().numpy()

                pert_losses.append(loss)

            pert = np.concatenate(pert_losses)

            delta_pct = 100.0 * (
                pert.mean() - clean[:, tq].mean()
            ) / max(clean[:, tq].mean(), 1e-12)

            deltas.append(delta_pct)

        rows.append({
            "target_channel": target,
            "grouped_all_other_full_mse_%": float(np.mean(deltas)),
        })

    return pd.DataFrame(rows)

grouped_frames = []

for d, H in CONDITIONS:
    protocol = load_condition_protocol(d, H)
    x, meta = load_and_normalize(d)
    X, Y = make_XY_from_origins(x, protocol["origins"], H)

    model, args, _ = train_or_load_timemixer(d, H)

    g = grouped_all_other_gain(
        model,
        X,
        Y,
        protocol["targets"],
        protocol["maps"],
        H,
        EPI_BATCH[d],
        max_donors=4,
    )

    g["dataset"] = d
    g["pred_len"] = int(H)
    grouped_frames.append(g)

    del model, X, Y
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

grouped_all_other = pd.concat(grouped_frames, ignore_index=True)
grouped_all_other.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_grouped_all_other.csv",
    index=False,
)

display(grouped_all_other.round(3))

grouped_summary = (
    grouped_all_other
    .groupby(["dataset", "pred_len"], as_index=False)
    .agg(
        targets=("target_channel", "size"),
        mean_grouped_effect_pct=("grouped_all_other_full_mse_%", "mean"),
        median_grouped_effect_pct=("grouped_all_other_full_mse_%", "median"),
        positive_target_fraction=(
            "grouped_all_other_full_mse_%",
            lambda s: float((np.asarray(s) > 0).mean()),
        ),
    )
)

print("\nGrouped all-other cross-channel usage:")
display(grouped_summary.round(3))


Reusing checkpoint: Electricity H=192 | test MSE=0.165758
Reusing checkpoint: Solar H=720 | test MSE=0.279617
Reusing checkpoint: ETTh1 H=720 | test MSE=0.565757


,target_channel,grouped_all_other_full_mse_%,dataset,pred_len
0,0,2.121,Electricity,192
1,64,26.842,Electricity,192
2,128,177.995,Electricity,192
3,192,136.894,Electricity,192
4,256,158.303,Electricity,192
5,320,221.549,Electricity,192
6,0,477.851,Solar,720
7,27,984.758,Solar,720
8,54,991.596,Solar,720
9,81,1076.412,Solar,720



Grouped all-other cross-channel usage:


,dataset,pred_len,targets,mean_grouped_effect_pct,median_grouped_effect_pct,positive_target_fraction
0,ETTh1,720,7,39.749,27.426,1.0
1,Electricity,192,6,120.618,147.599,1.0
2,Solar,720,6,960.808,1015.435,1.0



**Interpretation guard.**  
The grouped intervention is a predeclared sanity check, not a criterion for choosing another model.  
If TimeMixer-XC shows little or no cross-channel sensitivity, report that and do **not** replace it post hoc with a more favorable architecture.


## 11. Per-source EPI caches — exact same 16 donor mappings

In [12]:

def source_cache_path(dataset_name, H, source_idx):
    d = CACHE_DIR / f"{dataset_name}_H{H}"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"source_{int(source_idx):04d}.npz"

@torch.inference_mode()
def compute_source_cache(
    model,
    dataset_name,
    H,
    source_idx,
    X,
    Y,
    targets,
    donor_maps,
    clean,
    batch_size,
    donor_chunk,
):
    path = source_cache_path(dataset_name, H, source_idx)

    if path.exists():
        return True

    R, N = donor_maps.shape
    Tn = len(targets)

    delta_endpoint = np.zeros((R, N, Tn), dtype=np.float32)
    delta_full_mse = np.zeros((R, N, Tn), dtype=np.float32)
    delta_full_mae = np.zeros((R, N, Tn), dtype=np.float32)

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        B = end - start
        base_x = X[start:end]

        yb = torch.from_numpy(Y[start:end]).float().to(DEVICE)
        global_idx = np.arange(start, end, dtype=np.int64)

        for r_start in range(0, R, donor_chunk):
            r_idx = np.arange(
                r_start,
                min(r_start + donor_chunk, R),
                dtype=np.int64,
            )

            Q = len(r_idx)

            aug = np.broadcast_to(
                base_x[None],
                (
                    Q, B,
                    base_x.shape[1],
                    base_x.shape[2],
                ),
            ).copy()

            for q, r in enumerate(r_idx):
                donor_idx = donor_maps[r, global_idx]
                aug[q, :, :, source_idx] = (
                    X[donor_idx, :, source_idx]
                )

            aug_t = torch.from_numpy(
                aug.reshape(
                    Q * B,
                    aug.shape[2],
                    aug.shape[3],
                )
            ).float().to(DEVICE)

            pred = timemixer_forward(model, aug_t, H)

            pred = pred.reshape(
                Q, B, H, pred.shape[-1]
            )

            truth = yb[None, :, :, targets]
            err = pred[:, :, :, targets] - truth

            pert_endpoint = (
                err[:, :, -1, :]
                .square()
                .cpu()
                .numpy()
            )
            pert_full_mse = (
                err.square()
                .mean(dim=2)
                .cpu()
                .numpy()
            )
            pert_full_mae = (
                err.abs()
                .mean(dim=2)
                .cpu()
                .numpy()
            )

            for q, r in enumerate(r_idx):
                delta_endpoint[r, start:end] = (
                    pert_endpoint[q]
                    - clean["endpoint_mse"][start:end]
                )
                delta_full_mse[r, start:end] = (
                    pert_full_mse[q]
                    - clean["full_mse"][start:end]
                )
                delta_full_mae[r, start:end] = (
                    pert_full_mae[q]
                    - clean["full_mae"][start:end]
                )

            del aug, aug_t, pred, err
            del pert_endpoint, pert_full_mse, pert_full_mae

    np.savez_compressed(
        path,
        delta_endpoint_mse=delta_endpoint,
        delta_full_mse=delta_full_mse,
        delta_full_mae=delta_full_mae,
    )

    return False

def run_condition_cache(dataset_name, H):
    protocol = load_condition_protocol(dataset_name, H)

    x, meta = load_and_normalize(dataset_name)
    X, Y = make_XY_from_origins(
        x, protocol["origins"], H
    )

    model, args, result = train_or_load_timemixer(
        dataset_name, H
    )

    clean_file = (
        OUTPUT_DIR
        / f"{dataset_name}_H{H}_clean.npz"
    )

    if clean_file.exists():
        z = np.load(clean_file)
        clean = {
            "endpoint_mse": z["endpoint_mse"],
            "full_mse": z["full_mse"],
            "full_mae": z["full_mae"],
        }
        print("Reusing clean losses.")
    else:
        clean = clean_losses(
            model,
            X,
            Y,
            protocol["targets"],
            H,
            EPI_BATCH[dataset_name],
        )
        np.savez_compressed(clean_file, **clean)

    np.savez_compressed(
        OUTPUT_DIR / f"{dataset_name}_H{H}_protocol.npz",
        origins=protocol["origins"],
        targets=protocol["targets"],
        sources=protocol["sources"],
        shifts=protocol["shifts"],
        maps=protocol["maps"],
    )

    reused = 0

    for q, source_idx in enumerate(protocol["sources"], start=1):
        was_reused = compute_source_cache(
            model=model,
            dataset_name=dataset_name,
            H=H,
            source_idx=int(source_idx),
            X=X,
            Y=Y,
            targets=protocol["targets"],
            donor_maps=protocol["maps"],
            clean=clean,
            batch_size=EPI_BATCH[dataset_name],
            donor_chunk=DONOR_CHUNK[dataset_name],
        )

        reused += int(was_reused)

        print(
            f"  source {q}/{len(protocol['sources'])}: "
            f"{int(source_idx)} "
            f"{'reused' if was_reused else 'computed'}"
        )

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    print(
        f"{dataset_name} H={H} complete | "
        f"reused {reused}/{len(protocol['sources'])}"
    )

    del model, X, Y, x
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


In [13]:

for d, H in CONDITIONS:
    print("\n" + "=" * 100)
    print(f"TimeMixer-XC EPI | {d} H={H}")
    print("=" * 100)

    run_condition_cache(d, H)

print("\nAll TimeMixer-XC source caches complete.")



TimeMixer-XC EPI | Electricity H=192
Reusing checkpoint: Electricity H=192 | test MSE=0.165758
  source 1/132: 1 computed
  source 2/132: 8 computed
  source 3/132: 11 computed
  source 4/132: 13 computed
  source 5/132: 14 computed
  source 6/132: 23 computed
  source 7/132: 26 computed
  source 8/132: 29 computed
  source 9/132: 33 computed
  source 10/132: 34 computed
  source 11/132: 41 computed
  source 12/132: 42 computed
  source 13/132: 44 computed
  source 14/132: 46 computed
  source 15/132: 48 computed
  source 16/132: 51 computed
  source 17/132: 52 computed
  source 18/132: 55 computed
  source 19/132: 58 computed
  source 20/132: 63 computed
  source 21/132: 65 computed
  source 22/132: 68 computed
  source 23/132: 70 computed
  source 24/132: 71 computed
  source 25/132: 73 computed
  source 26/132: 77 computed
  source 27/132: 78 computed
  source 28/132: 83 computed
  source 29/132: 86 computed
  source 30/132: 87 computed
  source 31/132: 94 computed
  source 32/132:

## 12. Assemble TimeMixer EPI pair scores

In [14]:

METRICS = {
    "endpoint_mse": {
        "delta": "delta_endpoint_mse",
        "P": "P_endpoint_mse_%",
        "Ridge": "I_Ridge_endpoint_mse_%",
        "MLP": "I_MLP_endpoint_mse_%",
        "iTransformer": "EPI_endpoint_mse_%",
        "TimeMixer": "EPI_TimeMixerXC_endpoint_mse_%",
    },
    "full_mse": {
        "delta": "delta_full_mse",
        "P": "P_full_mse_%",
        "Ridge": "I_Ridge_full_mse_%",
        "MLP": "I_MLP_full_mse_%",
        "iTransformer": "EPI_full_mse_%",
        "TimeMixer": "EPI_TimeMixerXC_full_mse_%",
    },
    "full_mae": {
        "delta": "delta_full_mae",
        "P": "P_full_mae_%",
        "Ridge": "I_Ridge_full_mae_%",
        "MLP": "I_MLP_full_mae_%",
        "iTransformer": "EPI_full_mae_%",
        "TimeMixer": "EPI_TimeMixerXC_full_mae_%",
    },
}

def load_timemixer_arrays(dataset_name, H):
    proto = np.load(
        OUTPUT_DIR / f"{dataset_name}_H{H}_protocol.npz"
    )
    clean_z = np.load(
        OUTPUT_DIR / f"{dataset_name}_H{H}_clean.npz"
    )

    targets = proto["targets"].astype(np.int64)
    sources = proto["sources"].astype(np.int64)

    clean = {
        m: clean_z[m].astype(np.float32)
        for m in METRICS
    }

    delta = {}

    for metric, info in METRICS.items():
        blocks = []

        for source_idx in sources:
            z = np.load(
                source_cache_path(
                    dataset_name, H, int(source_idx)
                )
            )
            blocks.append(
                z[info["delta"]].astype(np.float32)
            )

        # [S,R,N,T] -> [R,N,T,S]
        delta[metric] = (
            np.stack(blocks, axis=0)
            .transpose(1, 2, 3, 0)
        )

    return {
        "targets": targets,
        "sources": sources,
        "origins": proto["origins"].astype(np.int64),
        "shifts": proto["shifts"].astype(np.int64),
        "clean": clean,
        "delta": delta,
    }

def importance_from_subset(
    delta,
    clean,
    donors=None,
    windows=None,
):
    R, N, Tn, Sn = delta.shape

    if donors is None:
        donors = np.arange(R)
    if windows is None:
        windows = np.arange(N)

    donors = np.asarray(donors, dtype=np.int64)
    windows = np.asarray(windows, dtype=np.int64)

    d = delta[
        np.ix_(
            donors,
            windows,
            np.arange(Tn),
            np.arange(Sn),
        )
    ]

    mean_delta = d.mean(axis=(0, 1))
    mean_clean = clean[windows].mean(axis=0)

    return (
        100.0
        * mean_delta
        / np.maximum(mean_clean[:, None], 1e-12)
    )

timemixer_arrays = {}
pair_frames = []

for d, H in CONDITIONS:
    arrays = load_timemixer_arrays(d, H)
    timemixer_arrays[(d, H)] = arrays

    p = previous[
        (previous["dataset"] == d)
        & (previous["pred_len"].astype(int) == int(H))
    ].copy()

    target_pos = {
        int(v): q
        for q, v in enumerate(arrays["targets"])
    }
    source_pos = {
        int(v): q
        for q, v in enumerate(arrays["sources"])
    }

    epi = {
        metric: importance_from_subset(
            arrays["delta"][metric],
            arrays["clean"][metric],
        )
        for metric in METRICS
    }

    rows = []

    for _, row in p.iterrows():
        rr = row.to_dict()

        tq = target_pos[int(row["target_channel"])]
        sq = source_pos[int(row["source_channel"])]

        for metric, info in METRICS.items():
            rr[info["TimeMixer"]] = float(
                epi[metric][tq, sq]
            )

        rows.append(rr)

    pair_frames.append(pd.DataFrame(rows))

timemixer_pairs = pd.concat(
    pair_frames,
    ignore_index=True,
)

if timesnet_previous is not None:
    tn_cols = [
        "dataset", "pred_len", "target_channel", "source_channel",
        "EPI_TimesNet_endpoint_mse_%",
        "EPI_TimesNet_full_mse_%",
        "EPI_TimesNet_full_mae_%",
    ]
    available = [c for c in tn_cols if c in timesnet_previous.columns]

    if len(available) == len(tn_cols):
        timemixer_pairs = timemixer_pairs.merge(
            timesnet_previous[tn_cols],
            on=[
                "dataset", "pred_len",
                "target_channel", "source_channel",
            ],
            how="left",
            validate="one_to_one",
        )

timemixer_pairs.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_timemixer_xc_epi_pair_scores.csv",
    index=False,
)

print("Pair rows:", len(timemixer_pairs))
display(timemixer_pairs.head(20).round(4))


Pair rows: 393


,dataset,pred_len,target_channel,source_channel,pool_tag,raw_abs_corr,P_endpoint_mse_%,P_full_mse_%,P_full_mae_%,I_iTransformer_endpoint_mse_%,...,I_MLP_full_mae_%,EPI_endpoint_mse_%,EPI_full_mse_%,EPI_full_mae_%,EPI_TimeMixerXC_endpoint_mse_%,EPI_TimeMixerXC_full_mse_%,EPI_TimeMixerXC_full_mae_%,EPI_TimesNet_endpoint_mse_%,EPI_TimesNet_full_mse_%,EPI_TimesNet_full_mae_%
0,Electricity,192,0,46,base,0.2108,-7.0641,-8.1682,-10.8208,-0.2519,...,2.7684,-0.2757,-0.2792,-0.1018,0.0856,0.0002,-0.5161,0.0888,0.0005,-0.0007
1,Electricity,192,0,123,base,0.2098,2.0731,0.4002,-2.1247,-0.2353,...,2.4436,-0.2203,-0.0592,-0.1479,1.1606,0.8092,0.3751,0.0258,-0.1231,-0.0460
2,Electricity,192,0,99,base,0.1871,0.1400,0.0073,-0.4590,-0.2365,...,1.0380,-0.1476,0.0415,-0.0710,1.1588,1.6849,1.1981,-0.3091,-0.1474,0.0030
3,Electricity,192,0,270,base,0.1800,-5.9889,-9.9914,-7.3117,-0.0147,...,1.7668,-0.1423,-0.1500,-0.1114,-0.2275,-0.5726,-0.2759,-0.4353,-0.1516,-0.0902
4,Electricity,192,0,127,base,0.1726,0.4396,-0.2725,-0.3610,-0.2163,...,1.7123,-0.2123,-0.0517,-0.0846,1.1720,0.6307,0.3699,-0.6820,-0.2978,-0.1362
5,Electricity,192,0,128,base,0.1599,-3.6407,-4.0292,-3.5608,-0.2075,...,0.7137,-0.2494,-0.1822,-0.1336,1.5867,0.7602,0.3903,-0.6212,-0.2567,-0.0346
6,Electricity,192,0,115,base,0.1580,-4.8527,-3.6146,-4.1619,-0.1362,...,-5.2323,-0.2956,-0.1317,-0.1206,-0.0195,-0.0039,-0.0181,-0.2946,-0.1229,0.0502
7,Electricity,192,0,130,base,0.1531,0.2202,-0.1331,-0.9530,-0.1584,...,0.2073,-0.1586,-0.0703,-0.1072,1.1796,0.8444,0.8555,0.2357,0.1532,0.0009
8,Electricity,192,0,219,base,0.0893,1.6380,1.2434,-1.6600,-0.0562,...,-0.5393,-0.1090,-0.1155,-0.0780,0.0169,0.2219,0.4473,-0.0024,-0.0026,-0.0364
9,Electricity,192,0,96,base,0.0173,0.9455,-0.2397,-1.0091,-0.3080,...,4.1116,-0.2543,-0.1501,-0.0835,-0.1204,0.0279,0.1405,0.0608,0.0291,-0.0266


## 13. Split-half reliability

In [15]:

def valid_source_mask(pair_df, target, sources):
    valid = set(
        pair_df[
            pair_df["target_channel"].astype(int)
            == int(target)
        ]["source_channel"]
        .astype(int)
        .tolist()
    )

    return np.asarray(
        [int(s) in valid for s in sources],
        dtype=bool,
    )

donor_rows = []
window_rows = []
temporal_rows = []

for d, H in CONDITIONS:
    a = timemixer_arrays[(d, H)]

    p = timemixer_pairs[
        (timemixer_pairs["dataset"] == d)
        & (timemixer_pairs["pred_len"].astype(int) == int(H))
    ]

    R = len(a["shifts"])
    N = len(a["origins"])

    rng_d = np.random.default_rng(
        SEED + H * 101 + 137
    )
    rng_w = np.random.default_rng(
        SEED + H * 211 + 173
    )

    temporal_blocks = [
        np.asarray(b, dtype=np.int64)
        for b in np.array_split(
            np.arange(N),
            N_TEMPORAL_BLOCKS,
        )
    ]

    for metric in METRICS:
        dlt = a["delta"][metric]
        cln = a["clean"][metric]

        for tq, target in enumerate(a["targets"]):
            mask = valid_source_mask(
                p, int(target), a["sources"]
            )

            if mask.sum() < 3:
                continue

            rhos = []
            for _ in range(N_SPLITS):
                perm = rng_d.permutation(R)
                A = perm[:R//2]
                B = perm[R//2:]

                IA = importance_from_subset(
                    dlt, cln, donors=A
                )[tq, mask]
                IB = importance_from_subset(
                    dlt, cln, donors=B
                )[tq, mask]

                rho = safe_spearman(IA, IB)
                if np.isfinite(rho):
                    rhos.append(rho)

            donor_rows.append({
                "dataset": d,
                "pred_len": int(H),
                "target_channel": int(target),
                "metric": metric,
                "median_rho": (
                    float(np.median(rhos))
                    if rhos else np.nan
                ),
                "mean_rho": (
                    float(np.mean(rhos))
                    if rhos else np.nan
                ),
            })

            rhos = []
            for _ in range(N_SPLITS):
                perm = rng_w.permutation(N)
                A = perm[:N//2]
                B = perm[N//2:]

                IA = importance_from_subset(
                    dlt, cln, windows=A
                )[tq, mask]
                IB = importance_from_subset(
                    dlt, cln, windows=B
                )[tq, mask]

                rho = safe_spearman(IA, IB)
                if np.isfinite(rho):
                    rhos.append(rho)

            window_rows.append({
                "dataset": d,
                "pred_len": int(H),
                "target_channel": int(target),
                "metric": metric,
                "median_rho": (
                    float(np.median(rhos))
                    if rhos else np.nan
                ),
                "mean_rho": (
                    float(np.mean(rhos))
                    if rhos else np.nan
                ),
            })

            block_scores = [
                importance_from_subset(
                    dlt, cln, windows=b
                )[tq, mask]
                for b in temporal_blocks
            ]

            rhos = []
            for i, j in itertools.combinations(
                range(len(block_scores)), 2
            ):
                rho = safe_spearman(
                    block_scores[i],
                    block_scores[j],
                )
                if np.isfinite(rho):
                    rhos.append(rho)

            temporal_rows.append({
                "dataset": d,
                "pred_len": int(H),
                "target_channel": int(target),
                "metric": metric,
                "mean_block_rho": (
                    float(np.mean(rhos))
                    if rhos else np.nan
                ),
            })

donor_target = pd.DataFrame(donor_rows)
window_target = pd.DataFrame(window_rows)
temporal_target = pd.DataFrame(temporal_rows)

def reliability_summary(df, value_col):
    return (
        df.groupby(
            ["dataset", "pred_len", "metric"],
            as_index=False,
        )
        .agg(
            targets=("target_channel", "size"),
            median_rho=(value_col, "median"),
            mean_rho=(value_col, "mean"),
        )
    )

donor_summary = reliability_summary(
    donor_target, "median_rho"
)
window_summary = reliability_summary(
    window_target, "median_rho"
)
temporal_summary = reliability_summary(
    temporal_target, "mean_block_rho"
)

donor_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_donor_reliability_summary.csv",
    index=False,
)
window_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_window_reliability_summary.csv",
    index=False,
)
temporal_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_temporal_reliability_summary.csv",
    index=False,
)

print("Donor split-half")
display(donor_summary.round(3))

print("Random-window split-half")
display(window_summary.round(3))

print("Temporal blocks")
display(temporal_summary.round(3))


Donor split-half


,dataset,pred_len,metric,targets,median_rho,mean_rho
0,ETTh1,720,endpoint_mse,7,0.943,0.829
1,ETTh1,720,full_mae,7,0.886,0.763
2,ETTh1,720,full_mse,7,0.886,0.824
3,Electricity,192,endpoint_mse,6,0.889,0.845
4,Electricity,192,full_mae,6,0.937,0.902
5,Electricity,192,full_mse,6,0.921,0.892
6,Solar,720,endpoint_mse,6,0.909,0.919
7,Solar,720,full_mae,6,0.956,0.952
8,Solar,720,full_mse,6,0.951,0.943


Random-window split-half


,dataset,pred_len,metric,targets,median_rho,mean_rho
0,ETTh1,720,endpoint_mse,7,0.943,0.910
1,ETTh1,720,full_mae,7,0.943,0.918
2,ETTh1,720,full_mse,7,0.943,0.935
3,Electricity,192,endpoint_mse,6,0.636,0.657
4,Electricity,192,full_mae,6,0.947,0.946
5,Electricity,192,full_mse,6,0.888,0.886
6,Solar,720,endpoint_mse,6,0.134,0.141
7,Solar,720,full_mae,6,0.889,0.857
8,Solar,720,full_mse,6,0.696,0.658


Temporal blocks


,dataset,pred_len,metric,targets,median_rho,mean_rho
0,ETTh1,720,endpoint_mse,7,0.381,0.343
1,ETTh1,720,full_mae,7,0.305,0.377
2,ETTh1,720,full_mse,7,0.429,0.427
3,Electricity,192,endpoint_mse,6,0.158,0.151
4,Electricity,192,full_mae,6,0.316,0.287
5,Electricity,192,full_mse,6,0.230,0.178
6,Solar,720,endpoint_mse,6,0.046,0.009
7,Solar,720,full_mae,6,0.388,0.342
8,Solar,720,full_mse,6,0.061,0.049


## 14. Alignment with utility and existing neural forecasters

In [16]:

score_map = {
    "Ridge": "I_Ridge_full_mse_%",
    "MLP": "I_MLP_full_mse_%",
    "iTransformer": "EPI_full_mse_%",
    "TimeMixer-XC": "EPI_TimeMixerXC_full_mse_%",
}

if "EPI_TimesNet_full_mse_%" in timemixer_pairs.columns:
    score_map["TimesNet"] = "EPI_TimesNet_full_mse_%"

target_rows = []

for (d, H, target), sub_all in timemixer_pairs.groupby(
    ["dataset", "pred_len", "target_channel"]
):
    # Primary pool is Extended, matching the current manuscript.
    sub = sub_all.copy()

    if len(sub) < 3:
        continue

    ids = sub["source_channel"].to_numpy()
    k = min(RANK_K, len(sub))

    row = {
        "dataset": d,
        "pred_len": int(H),
        "target_channel": int(target),
        "n_sources": len(sub),
        "rho_P_TimeMixer": safe_spearman(
            sub["P_full_mse_%"],
            sub["EPI_TimeMixerXC_full_mse_%"],
        ),
        "jaccard_P_TimeMixer": topk_jaccard(
            sub["P_full_mse_%"],
            sub["EPI_TimeMixerXC_full_mse_%"],
            ids,
            k,
        ),
    }

    for name, col in score_map.items():
        if name == "TimeMixer-XC":
            continue

        row[f"rho_{name}_TimeMixer"] = safe_spearman(
            sub[col],
            sub["EPI_TimeMixerXC_full_mse_%"],
        )

        row[f"jaccard_{name}_TimeMixer"] = topk_jaccard(
            sub[col],
            sub["EPI_TimeMixerXC_full_mse_%"],
            ids,
            k,
        )

    target_rows.append(row)

alignment_target = pd.DataFrame(target_rows)
alignment_target.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_alignment_target.csv",
    index=False,
)

summary_items = {
    "targets": len(alignment_target),
    "median_rho_P_TimeMixer": alignment_target["rho_P_TimeMixer"].median(),
    "mean_rho_P_TimeMixer": alignment_target["rho_P_TimeMixer"].mean(),
    "mean_jaccard_P_TimeMixer": alignment_target["jaccard_P_TimeMixer"].mean(),
}

for name in score_map:
    if name == "TimeMixer-XC":
        continue

    c = f"rho_{name}_TimeMixer"
    j = f"jaccard_{name}_TimeMixer"

    if c in alignment_target.columns:
        summary_items[f"median_rho_{name}_TimeMixer"] = alignment_target[c].median()
        summary_items[f"mean_rho_{name}_TimeMixer"] = alignment_target[c].mean()
        summary_items[f"mean_jaccard_{name}_TimeMixer"] = alignment_target[j].mean()

alignment_summary = pd.DataFrame([summary_items])
alignment_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_alignment_summary.csv",
    index=False,
)

display(alignment_target.round(3))
print("\nPRIMARY FULL-MSE SUMMARY")
display(alignment_summary.round(3))


,dataset,pred_len,target_channel,n_sources,rho_P_TimeMixer,jaccard_P_TimeMixer,rho_Ridge_TimeMixer,jaccard_Ridge_TimeMixer,rho_MLP_TimeMixer,jaccard_MLP_TimeMixer,rho_iTransformer_TimeMixer,jaccard_iTransformer_TimeMixer,rho_TimesNet_TimeMixer,jaccard_TimesNet_TimeMixer
0,ETTh1,720,0,6,0.143,0.667,0.829,0.667,0.886,1.000,-0.143,0.667,0.257,0.667
1,ETTh1,720,1,6,-0.771,0.667,0.657,0.667,-0.829,0.667,-0.257,0.667,0.600,0.667
2,ETTh1,720,2,6,-0.257,0.667,0.486,0.667,0.486,0.667,0.143,0.667,0.600,1.000
3,ETTh1,720,3,6,-0.943,0.667,0.714,1.000,-0.657,0.667,-0.257,0.667,-0.429,0.667
4,ETTh1,720,4,6,-0.886,0.667,0.657,1.000,0.657,1.000,0.371,0.667,0.543,1.000
5,ETTh1,720,5,6,0.600,0.667,0.714,1.000,0.200,0.667,-0.371,0.667,-0.029,0.667
6,ETTh1,720,6,6,0.543,0.667,-0.771,0.667,-0.600,0.667,-0.657,0.667,-0.314,0.667
7,Electricity,192,0,28,0.573,0.111,-0.118,0.000,0.148,0.111,0.172,0.111,-0.165,0.000
8,Electricity,192,64,30,0.371,0.250,0.318,0.250,0.149,0.000,0.238,0.111,0.103,0.000
9,Electricity,192,128,28,0.222,0.111,0.404,0.000,-0.054,0.000,-0.009,0.111,0.034,0.111



PRIMARY FULL-MSE SUMMARY


,targets,median_rho_P_TimeMixer,mean_rho_P_TimeMixer,mean_jaccard_P_TimeMixer,median_rho_Ridge_TimeMixer,mean_rho_Ridge_TimeMixer,mean_jaccard_Ridge_TimeMixer,median_rho_MLP_TimeMixer,mean_rho_MLP_TimeMixer,mean_jaccard_MLP_TimeMixer,median_rho_iTransformer_TimeMixer,mean_rho_iTransformer_TimeMixer,mean_jaccard_iTransformer_TimeMixer,median_rho_TimesNet_TimeMixer,mean_rho_TimesNet_TimeMixer,mean_jaccard_TimesNet_TimeMixer
0,19,0.053,-0.054,0.3,0.318,0.291,0.374,0.148,0.074,0.31,-0.009,-0.032,0.301,0.103,0.1,0.357


## 15. Three-neural-forecaster correlation matrix

In [17]:

neural_names = ["iTransformer", "TimeMixer-XC"]

if "TimesNet" in score_map:
    neural_names.insert(1, "TimesNet")

matrix = pd.DataFrame(
    np.eye(len(neural_names)),
    index=neural_names,
    columns=neural_names,
    dtype=float,
)

pairwise_rows = []

for A, B in itertools.combinations(neural_names, 2):
    colA = score_map[A]
    colB = score_map[B]

    rhos = []
    jaccards = []

    for (_, _, _), sub in timemixer_pairs.groupby(
        ["dataset", "pred_len", "target_channel"]
    ):
        if len(sub) < 3:
            continue

        rho = safe_spearman(sub[colA], sub[colB])
        if np.isfinite(rho):
            rhos.append(rho)

        ids = sub["source_channel"].to_numpy()
        k = min(RANK_K, len(sub))

        jaccards.append(
            topk_jaccard(
                sub[colA], sub[colB], ids, k
            )
        )

    med = float(np.median(rhos)) if rhos else np.nan
    mean = float(np.mean(rhos)) if rhos else np.nan
    jac = float(np.mean(jaccards)) if jaccards else np.nan

    matrix.loc[A, B] = med
    matrix.loc[B, A] = med

    pairwise_rows.append({
        "forecaster_A": A,
        "forecaster_B": B,
        "targets": len(rhos),
        "median_rho": med,
        "mean_rho": mean,
        "mean_top5_jaccard": jac,
    })

pairwise_neural = pd.DataFrame(pairwise_rows)

pairwise_neural.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_three_neural_alignment.csv",
    index=False,
)

print("Median target-wise full-MSE EPI Spearman")
display(matrix.round(3))

print("Pairwise details")
display(pairwise_neural.round(3))


Median target-wise full-MSE EPI Spearman


,iTransformer,TimesNet,TimeMixer-XC
iTransformer,1.000,-0.091,-0.009
TimesNet,-0.091,1.000,0.103
TimeMixer-XC,-0.009,0.103,1.000


Pairwise details


,forecaster_A,forecaster_B,targets,median_rho,mean_rho,mean_top5_jaccard
0,iTransformer,TimesNet,19,-0.091,0.045,0.348
1,iTransformer,TimeMixer-XC,19,-0.009,-0.032,0.301
2,TimesNet,TimeMixer-XC,19,0.103,0.100,0.357


## 16. Primary reliability overview

In [18]:

rows = []

for d, H in CONDITIONS:
    def lookup(df):
        q = df[
            (df["dataset"] == d)
            & (df["pred_len"].astype(int) == int(H))
            & (df["metric"] == "full_mse")
        ]

        return (
            float(q.iloc[0]["median_rho"])
            if len(q) == 1
            else np.nan
        )

    rows.append({
        "dataset": d,
        "pred_len": int(H),
        "donor_split_half_rho": lookup(donor_summary),
        "random_window_split_half_rho": lookup(window_summary),
        "temporal_block_rho": lookup(temporal_summary),
    })

reliability_overview = pd.DataFrame(rows)
reliability_overview.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_full_mse_reliability_overview.csv",
    index=False,
)

display(reliability_overview.round(3))


,dataset,pred_len,donor_split_half_rho,random_window_split_half_rho,temporal_block_rho
0,Electricity,192,0.921,0.888,0.230
1,Solar,720,0.951,0.696,0.061
2,ETTh1,720,0.886,0.943,0.429


## 17. Conservative automatic interpretation

In [19]:

print("=" * 100)
print("THIRD-ARCHITECTURE FUNCTIONAL-RELIANCE TEST")
print("=" * 100)

print("\nDirect model:")
display(training_summary[
    ["dataset", "pred_len", "test_mse", "test_mae", "best_epoch"]
].round(6))

print("\nGrouped all-other cross-channel effect:")
display(grouped_summary.round(3))

print("\nFunctional-reliance reliability:")
display(reliability_overview.round(3))

print("\nAlignment summary:")
display(alignment_summary.round(3))

if len(pairwise_neural):
    print("\nThree-neural-forecaster alignment:")
    display(pairwise_neural.round(3))

rho_p = float(
    alignment_summary.iloc[0]["median_rho_P_TimeMixer"]
)

print("\nInterpretation constraints:")

if np.isfinite(rho_p) and abs(rho_p) < 0.30:
    print(
        "- TimeMixer-XC functional reliance is weakly aligned with "
        "controlled predictive utility."
    )
else:
    print(
        "- TimeMixer-XC shows non-negligible alignment with controlled "
        "predictive utility; report the measured alignment directly."
    )

reliable = (
    (reliability_overview["donor_split_half_rho"] >= 0.60)
    & (reliability_overview["random_window_split_half_rho"] >= 0.60)
)

print(
    f"- Reliable full-MSE conditions: "
    f"{int(reliable.sum())}/{len(reliable)}"
)

positive_group = (
    grouped_summary["median_grouped_effect_pct"] > 0
)

print(
    f"- Conditions with positive median grouped all-other effect: "
    f"{int(positive_group.sum())}/{len(positive_group)}"
)

print("\nWording guide:")
print(
    "Do NOT claim that every architecture must have a different graph."
)
print(
    "Use: the evaluated neural forecasters do not support a single "
    "architecture-independent functional-reliance ranking."
)
print(
    "If two architectures align but the third does not, that still supports "
    "architecture conditioning rather than universal disagreement."
)
print(
    "If TimeMixer-XC is unreliable, do not use its pairwise ranking as a "
    "strong confirmatory result; report the reliability limitation."
)

if RUN_MODE == "screen":
    print("\nNEXT:")
    print(
        "If the notebook executes correctly, set RUN_MODE='full' and rerun. "
        "Do this regardless of whether the ETTh1 screen is favorable."
    )


THIRD-ARCHITECTURE FUNCTIONAL-RELIANCE TEST

Direct model:


,dataset,pred_len,test_mse,test_mae,best_epoch
0,Electricity,192,0.165758,0.266455,14
1,Solar,720,0.279617,0.306191,2
2,ETTh1,720,0.565757,0.535723,2



Grouped all-other cross-channel effect:


,dataset,pred_len,targets,mean_grouped_effect_pct,median_grouped_effect_pct,positive_target_fraction
0,ETTh1,720,7,39.749,27.426,1.0
1,Electricity,192,6,120.618,147.599,1.0
2,Solar,720,6,960.808,1015.435,1.0



Functional-reliance reliability:


...


Alignment summary:


,targets,median_rho_P_TimeMixer,mean_rho_P_TimeMixer,mean_jaccard_P_TimeMixer,median_rho_Ridge_TimeMixer,mean_rho_Ridge_TimeMixer,mean_jaccard_Ridge_TimeMixer,median_rho_MLP_TimeMixer,mean_rho_MLP_TimeMixer,mean_jaccard_MLP_TimeMixer,median_rho_iTransformer_TimeMixer,mean_rho_iTransformer_TimeMixer,mean_jaccard_iTransformer_TimeMixer,median_rho_TimesNet_TimeMixer,mean_rho_TimesNet_TimeMixer,mean_jaccard_TimesNet_TimeMixer
0,19,0.053,-0.054,0.3,0.318,0.291,0.374,0.148,0.074,0.31,-0.009,-0.032,0.301,0.103,0.1,0.357



Three-neural-forecaster alignment:


,forecaster_A,forecaster_B,targets,median_rho,mean_rho,mean_top5_jaccard
0,iTransformer,TimesNet,19,-0.091,0.045,0.348
1,iTransformer,TimeMixer-XC,19,-0.009,-0.032,0.301
2,TimesNet,TimeMixer-XC,19,0.103,0.100,0.357



Interpretation constraints:
- TimeMixer-XC functional reliance is weakly aligned with controlled predictive utility.
- Reliable full-MSE conditions: 3/3
- Conditions with positive median grouped all-other effect: 3/3

Wording guide:
Do NOT claim that every architecture must have a different graph.
Use: the evaluated neural forecasters do not support a single architecture-independent functional-reliance ranking.
If two architectures align but the third does not, that still supports architecture conditioning rather than universal disagreement.
If TimeMixer-XC is unreliable, do not use its pairwise ranking as a strong confirmatory result; report the reliability limitation.
